#### READ THE PRODUCTS DATA FROM VOLUME

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/Products/'
CHECKPOINT_DIR = '/Volumes/orders_sales/landing/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/products'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'PRODUCTS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
products_schema = """product_id STRING, 
                     product_category_name STRING,
                     product_name_lenght STRING,
                     product_description_lenght STRING,
                     product_photos_qty STRING,
                     product_weight_g STRING,
                     product_length_cm STRING,
                     product_height_cm STRING,
                     product_width_cm STRING
                 """

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.PRODUCTS`

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

products_df  = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(products_schema)
                 .load(f'{BASE_DIR}')
        )
products_trans_df = products_df.select(
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght", 
    "product_photos_qty", 
    "product_weight_g", 
    "product_length_cm", 
    "product_height_cm", 
    "product_width_cm",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

# products_trans_df.printSchema()
# products_trans_df.limit(1).display()

qry = (
    products_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()